# Table Transformer Pipeline → Markdown

Notebook para detección de tablas y reconstrucción en markdown usando Table Transformer (TATR).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from transformers import (
    AutoImageProcessor,
    TableTransformerForObjectDetection
)

import pytesseract
from typing import Any, Dict, Iterable, List, Optional, Tuple


c:\Users\user\proyectos\vision-docs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        markers = [
            ("data", "notebooks"),
            ("src", "data"),
            ("pyproject.toml",),
            ("requirements.txt",),
        ]
        if any(all((base / m).exists() for m in group) for group in markers):
            return base
    return start

In [3]:
# ============================================================
# Configuración
# ============================================================

RUN_NAME = "paper_run_001"

PROJECT_ROOT = find_project_root()

RUN_DIR = PROJECT_ROOT / "data" / "outputs" / RUN_NAME

INPUTS = {
    "pages": RUN_DIR / "01_page_images",
    "manifest": RUN_DIR / "11_manifests" / "detections.csv"
}

OUTPUTS = {
    "tables": RUN_DIR / "21_tables_tatr",
    "markdown": RUN_DIR / "22_tables_markdown"
}

OUTPUTS["tables"].mkdir(parents=True, exist_ok=True)
OUTPUTS["markdown"].mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)


DEVICE: cpu


In [4]:
# ============================================================
# Leer detecciones
# ============================================================

df = pd.read_csv(INPUTS["manifest"])

print(df.head())
print(df.columns)


   page_num  kind                     bbox     score       source meta
0         1  text   (424, 934, 1479, 1330)  0.993083  ppstructure  NaN
1         1  text  (427, 1373, 1478, 1438)  0.960642  ppstructure  NaN
2         1  text   (319, 2225, 926, 2291)  0.931423  ppstructure  NaN
3         1  text  (421, 1664, 1482, 1807)  0.930575  ppstructure  NaN
4         1  text    (430, 713, 1473, 744)  0.910238  ppstructure  NaN
Index(['page_num', 'kind', 'bbox', 'score', 'source', 'meta'], dtype='object')


In [5]:
# ============================================================
# Filtrar tablas
# ============================================================

tables_df = df[df["kind"] == "table"].copy()

print("Cantidad tablas:", len(tables_df))

tables_df.head()


Cantidad tablas: 1


,page_num,kind,bbox,score,source,meta
13,3,table,"(302, 398, 1406, 698)",0.999671,tatr,{'label': 0}


In [6]:
# ============================================================
# Cargar modelos TATR
# ============================================================

structure_processor = AutoImageProcessor.from_pretrained(
    "microsoft/table-transformer-structure-recognition"
)

structure_model = TableTransformerForObjectDetection.from_pretrained(
    "microsoft/table-transformer-structure-recognition"
).to(DEVICE)

detection_processor = AutoImageProcessor.from_pretrained(
    "microsoft/table-transformer-detection"
)

detection_model = TableTransformerForObjectDetection.from_pretrained(
    "microsoft/table-transformer-detection"
).to(DEVICE)

print("TATR cargado")


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.
Some weights of the model checkpoint at microsoft/table-transformer-structure-recognition were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClass

TATR cargado


In [7]:
# ============================================================
# Dibujar boxes
# ============================================================

def draw_boxes(image, results):

    img = image.copy()

    for score, label, box in zip(
        results["scores"],
        results["labels"],
        results["boxes"]
    ):

        box = [int(i) for i in box.tolist()]

        cv2.rectangle(
            img,
            (box[0], box[1]),
            (box[2], box[3]),
            (0, 255, 0),
            2
        )

        cv2.putText(
            img,
            str(label.item()),
            (box[0], box[1] - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            1
        )

    return img


In [8]:
def ocr_cell(cell_img):

    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)

    # denoise
    gray = cv2.fastNlMeansDenoising(gray)

    # CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    gray = clahe.apply(gray)

    # binarización
    gray = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        11
    )

    # bordes
    gray = cv2.copyMakeBorder(
        gray,
        10,10,10,10,
        cv2.BORDER_CONSTANT,
        value=255
    )

    text = pytesseract.image_to_string(
        gray,
        config="--oem 3 --psm 6 -l spa+eng"
    )

    return text.strip()

In [9]:
# ============================================================
# Markdown
# ============================================================

def dataframe_to_markdown(df):

    return df.to_markdown(index=False)


In [10]:
# ============================================================
# Detectar estructura
# ============================================================

def detect_table_structure(table_image):

    pil_image = Image.fromarray(
        cv2.cvtColor(table_image, cv2.COLOR_BGR2RGB)
    )

    encoding = structure_processor(
        images=pil_image,
        return_tensors="pt"
    )

    encoding = {k: v.to(DEVICE) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = structure_model(**encoding)

    target_sizes = torch.tensor([pil_image.size[::-1]]).to(DEVICE)

    results = structure_processor.post_process_object_detection(
        outputs,
        threshold=0.3,
        target_sizes=target_sizes
    )[0]

    return results


In [11]:
# ============================================================
# Labels
# ============================================================

id2label = structure_model.config.id2label

print(id2label)


{0: 'table', 1: 'table column', 2: 'table row', 3: 'table column header', 4: 'table projected row header', 5: 'table spanning cell'}


In [12]:
def reconstruct_table(table_image, results):

    rows = []
    cols = []

    # ---------------------------------------------------
    # extraer rows y columns
    # ---------------------------------------------------

    for score, label, box in zip(
        results["scores"],
        results["labels"],
        results["boxes"]
    ):

        label_name = id2label[label.item()]

        box = [int(i) for i in box.tolist()]

        if label_name == "table row":
            rows.append(box)

        elif label_name == "table column":
            cols.append(box)

    # ordenar
    rows = sorted(rows, key=lambda x: x[1])
    cols = sorted(cols, key=lambda x: x[0])

    print("ROWS:", len(rows))
    print("COLS:", len(cols))

    # ---------------------------------------------------
    # construir matriz
    # ---------------------------------------------------

    matrix = []

    for r in rows:

        row_data = []

        ry1, ry2 = r[1], r[3]

        for c in cols:

            cx1, cx2 = c[0], c[2]

            # usar TODA la altura de la fila
            # y TODO el ancho de la columna

            pad = 5

            crop = table_image[
                max(0, ry1+pad):min(table_image.shape[0], ry2-pad),
                max(0, cx1+pad):min(table_image.shape[1], cx2-pad)
            ]

            if crop.size == 0:
                row_data.append("")
                continue

            text = ocr_cell(crop)

            row_data.append(text)

        matrix.append(row_data)

    # ---------------------------------------------------
    # dataframe
    # ---------------------------------------------------

    df = pd.DataFrame(matrix)

    # limpiar
    df = df.replace("", np.nan)

    df = df.dropna(how="all")

    df = df.dropna(axis=1, how="all")

    return df

In [13]:
import ast

tables_df["bbox"] = tables_df["bbox"].apply(ast.literal_eval)

In [14]:
# ============================================================
# Pipeline completo
# ============================================================

all_markdowns = []

for idx, row in tables_df.iterrows():

    # ----------------------------
    # Datos reales del manifest
    # ----------------------------
    page_num = int(row["page_num"])

    x1, y1, x2, y2 = row["bbox"]

    # ----------------------------
    # Ruta imagen página
    # ----------------------------
    page_path = INPUTS["pages"] / f"page_{page_num:03d}.png"

    image = cv2.imread(str(page_path))

    if image is None:
        print("No se pudo leer:", page_path)
        continue

    # ----------------------------
    # Recorte tabla
    # ----------------------------
    pad = 40

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)

    x2 = min(image.shape[1], x2 + pad)
    y2 = min(image.shape[0], y2 + pad)

    table_crop = image[y1:y2, x1:x2]

    if table_crop.size == 0:
        print(f"Tabla vacía {idx}")
        continue

    # ----------------------------
    # Upscaling
    # ----------------------------
    table_crop = cv2.resize(
        table_crop,
        None,
        fx=4,
        fy=4,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.cvtColor(table_crop, cv2.COLOR_BGR2GRAY)

    gray = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    table_crop = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) 

    # ----------------------------
    # TATR
    # ----------------------------
    results = detect_table_structure(table_crop)

    # ----------------------------
    # Debug visual
    # ----------------------------
    debug = draw_boxes(table_crop.copy(), results)

    debug_path = OUTPUTS["tables"] / f"table_{idx:04d}.png"

    cv2.imwrite(str(debug_path), debug)

    try:

        # ----------------------------
        # Reconstrucción
        # ----------------------------
        df_table = reconstruct_table(table_crop, results)
        print(df_table.shape)
        print(df_table.head())

        markdown = dataframe_to_markdown(df_table)

        # ----------------------------
        # Guardar markdown
        # ----------------------------
        md_path = OUTPUTS["markdown"] / f"table_{idx:04d}.md"

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(markdown)

        all_markdowns.append({
            "table_id": idx,
            "markdown": markdown
        })

        print(f"Tabla {idx} OK")

    except Exception as e:

        print(f"Error tabla {idx}: {e}")

ROWS: 5
COLS: 4
(5, 4)
            0                                   1  \
0  Retrovirus  Negativos a\ndermatofitosis\n% (n)   
1   VLeF -VIF                                 NaN   
2        VLeF                            20.0 (7)   
3         VIF                            11.4 (4)   
4       Total                           31.4 (11)   

                                    2             3  
0  Positivos a\ndermatofitosis\n% (n)  Total\n% (n)  
1                             5.7 (2)       5.7 (2)  
2                           48.6 (17)     68.6 (24)  
3                            14.3 (5)      25.7 (9)  
4                           68.6 (24)      100 (35)  
Tabla 13 OK


In [15]:
print(all_markdowns[0]["markdown"])

| 0          | 1              | 2              | 3         |
|:-----------|:---------------|:---------------|:----------|
| Retrovirus | Negativos a    | Positivos a    | Total     |
|            | dermatofitosis | dermatofitosis | % (n)     |
|            | % (n)          | % (n)          |           |
| VLeF -VIF  | nan            | 5.7 (2)        | 5.7 (2)   |
| VLeF       | 20.0 (7)       | 48.6 (17)      | 68.6 (24) |
| VIF        | 11.4 (4)       | 14.3 (5)       | 25.7 (9)  |
| Total      | 31.4 (11)      | 68.6 (24)      | 100 (35)  |
